In [1]:
import os
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count=2"

from jVMC_exp.sharding_config import MESH
import jax

print("Devices =", jax.devices())
print("Device count =", jax.device_count())
print("Mesh shape =", MESH.shape["devices"])

Running in single-node mode (JVMC_USE_DISTRIBUTED not set)
Devices = [CpuDevice(id=0), CpuDevice(id=1)]
Device count = 2
Mesh shape = 2


In [2]:
import jax
# jax.config.update("jax_log_compiles", True)
import jax.numpy as jnp

from jVMC_exp.vqs import NQS
from jVMC_exp.sharding_config import sharded
import jVMC_exp


In [3]:
class TestNQS(NQS):
    @sharded(automatic_sharding=True, yield_iter=True)
    def _gradients_iter_sh(self, s, *, parameters, batch_size):
        return self.flat_gradient_function(self.apply_fun, parameters, s)

    def iterative_gradients(self, s):
        return self._gradients_iter_sh(s, parameters=self.grad_parameters, batch_size=self.batchSize)

In [4]:
L = 10
n_samples = 2**8
n_chains = 2**6
batch_size = 2**4

model = jVMC_exp.nets.RBM(L)
psi = NQS(model, L, batch_size, seed=123)
sampler = jVMC_exp.sampler.MCSampler(
    psi, 
    jVMC_exp.propose.SpinFlip(),
    123, 
    n_chains,
    n_samples    
)

samples = sampler.samples

In [5]:
len(psi.iterative_gradients(samples))

16

In [7]:
gradients = []
iterable = psi.iterative_gradients(samples)

for grad in iterable:
    gradients.append(grad)
gradients = jnp.concatenate(gradients)

jnp.allclose(gradients, psi.gradients(samples))

Array(True, dtype=bool)

In [ ]:
from chex import dataclass
from typing import Iterable, Iterator

class ReusableIterable():
    def __init__(self, iterable: Iterable, n_iterations: None | int = None):
        self.
        
    def __len__(self):
        return self.n_iterations

    def __iter__(self) -> Iterator:
        return iter(self.iterable)
    



In [26]:
gradients.addressable_shards

[Shard(device=CpuDevice(id=0), index=(slice(0, 128, None), slice(None, None, None)), replica_id=0, data=[[ 0.91607644+0.j  0.90918861+0.j -0.1580211 +0.j ...  0.84259112+0.j
    0.87658272+0.j -0.97292724+0.j]
  [ 0.74602473+0.j  0.44976429+0.j -0.96731269+0.j ... -0.88404705+0.j
    0.31211205+0.j  0.92885278+0.j]
  [ 0.90403043+0.j -0.6130788 +0.j -0.2383867 +0.j ...  0.94695684+0.j
   -0.93032902+0.j  0.0773387 +0.j]
  ...
  [ 0.66415854+0.j -0.72333738+0.j  0.96196209+0.j ... -0.73167764+0.j
    0.91413325+0.j -0.6743908 +0.j]
  [ 0.65634846+0.j  0.8947671 +0.j -0.86052917+0.j ... -0.80174442+0.j
    0.82866016+0.j  0.79971566+0.j]
  [ 0.93334726+0.j  0.62232708+0.j -0.9749899 +0.j ... -0.64021336+0.j
    0.12308659+0.j  0.62173325+0.j]]),
 Shard(device=CpuDevice(id=1), index=(slice(128, 256, None), slice(None, None, None)), replica_id=0, data=[[-0.63648388+0.j -0.85953657+0.j -0.25400285+0.j ...  0.21953832+0.j
    0.84968045+0.j -0.96809436+0.j]
  [ 0.56628692+0.j  0.0641997 +0.j